In [7]:
! pip install pyhive thrift thrift_sasl

  Using cached thrift_sasl-0.4.3-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached pure_sasl-0.6.2-py3-none-any.whl
Using cached thrift_sasl-0.4.3-py2.py3-none-any.whl (8.3 kB)


In [12]:
from pyhive import hive

conn = hive.Connection(
    host="spark-thrift",       # or spark-thriftserver hostname
    port=10000,
    username="airflow",
)

cursor = conn.cursor()

cursor.execute("SHOW DATABASES")

for row in cursor.fetchall():
    print(row)

TTransportException: unexpected exception

## dbt model excerpts
Below we load and display the SQL for the bronze staging model and the silver model involved in the failure (`stg_batch__balances` → `silver_daily_account_balance`).

In [14]:
from pathlib import Path
repo_root = Path('..').resolve()  # notebook sits in `notebooks/`
models_dir = repo_root / 'dbt' / 'banking_analytics' / 'models'
silver_path = models_dir / 'silver' / 'silver_daily_account_balance.sql'
bronze_path = models_dir / 'bronze' / 'stg_batch__balances.sql'
for p in (silver_path, bronze_path):
    print('\
' + '='*80)
    print(p)
    if p.exists():
        print(p.read_text())
    else:
        print('NOT FOUND:', p)

/home/jovyan/dbt/banking_analytics/models/silver/silver_daily_account_balance.sql
NOT FOUND: /home/jovyan/dbt/banking_analytics/models/silver/silver_daily_account_balance.sql
/home/jovyan/dbt/banking_analytics/models/bronze/stg_batch__balances.sql
NOT FOUND: /home/jovyan/dbt/banking_analytics/models/bronze/stg_batch__balances.sql


## Check Spark catalog and table visibility
Use the Thrift connection to inspect the `analytics` schema and attempt a simple query against the bronze table. This shows whether the table is registered in the metastore (the likely cause if the silver model cannot resolve the ref).

In [15]:
from pyhive import hive
from TCLIService.ttypes import TOperationState
import traceback

def run_query(q):
    conn = hive.Connection(host='spark-thrift', port=10000, username='airflow')
    cur = conn.cursor()
    try:
        cur.execute(q)
        return cur.fetchall()
    except Exception as e:
        print('Query failed:', q)
        traceback.print_exc()
        return None

print('Show tables in analytics:')
tables = run_query('SHOW TABLES IN analytics')
print(tables)

print('
Attempt describe or count on analytics.stg_batch__balances:')
res = run_query('DESCRIBE TABLE analytics.stg_batch__balances')
print(res)

# Try a count (may be expensive)
try:
    cnt = run_query('SELECT count(*) FROM analytics.stg_batch__balances')
    print('count result:', cnt)
except Exception:
    print('Count failed or table not found')

SyntaxError: unterminated string literal (detected at line 20) (3977737870.py, line 20)

## Quick troubleshooting notes
If the `DESCRIBE` or `SHOW TABLES` do not list `analytics.stg_batch__balances`, common causes are:
- The bronze model was not run in the same metastore/session (it created files in S3 but did not register a table).
- The bronze table was created under a different catalog/schema than the silver model expects. Check `profiles.yml` schema and any `+schema` overrides.
- The silver run selected only `silver_daily_account_balance` (no parents). Run with its ancestors so staging models are built or ensure the staging table exists in the metastore.

Recommended commands to run in the Airflow/dbt container to resolve the issue:
```bash
# Build staging (bronze) and silver together (include ancestors)
dbt run --project-dir /opt/airflow/dbt/banking_analytics --profiles-dir /opt/airflow/dbt --target spark --select +silver_daily_account_balance

# Or run the specific staging model first, then the silver model
dbt run --project-dir /opt/airflow/dbt/banking_analytics --profiles-dir /opt/airflow/dbt --target spark --select stg_batch__balances
dbt run --project-dir /opt/airflow/dbt/banking_analytics --profiles-dir /opt/airflow/dbt --target spark --select silver_daily_account_balance
```

If the bronze model writes Delta files to S3 but the table is not registered in the metastore, register it manually or change the bronze materialization to create a managed/external table registered in the metastore (ensure `location_root` is applied and the adapter creates the table).